In [0]:
from pyspark.sql.functions import max, col

In [0]:
def silver_incremental_ingest(src_bronze_path, tgt_silver_table):

    # Read bronze data
    raw_df = spark.read \
        .format("delta") \
        .load(src_bronze_path)

    # Check if silver table exists
    if spark.catalog.tableExists(tgt_silver_table):

        silver_df = spark.table(tgt_silver_table)

        # Get max timestamp from silver
        max_ts_row = silver_df.select(max("load_timestamp")).collect()[0]
        max_ts = max_ts_row[0]

        if max_ts is None:
            print("Silver table is empty. Will load all records.")

    else:
        print("Silver table not found. Will load all records (initial load).")
        max_ts = None

    # Incremental filter
    if max_ts:
        df = raw_df.filter(col("load_timestamp") > max_ts)
    else:
        df = raw_df

    print(f"Number of records to load: {df.count()}")

    return df